# Lie Symmetry Analysis of the Porous Medium Equation

This tutorial analyzes the nonlinear degenerate diffusion equation
$$u_t - (u u_x)_x = u_t - u_x^2 - u u_{xx} = 0$$
Since $(u u_x)_x = \tfrac12 (u^2)_{xx}$, this is the standard one-dimensional porous-medium equation with exponent $m=2$, up to a constant rescaling of time. In the alternative diffusivity convention $u_t=(u^m u_x)_x$, the diffusivity exponent is $m=1$.

We investigate:
1. The 4-dimensional Lie point symmetry group (translations, kinematic scaling, Barenblatt field dilation).
2. Verification of infinitesimal generators.
3. Symmetry reduction to the famous compact-support **Barenblatt–Pattle self-similar source solution**.

In [ ]:
import sympy as sp

from symlie import (
    infinitesimals,
    lie_bracket,
    max_derivative_order,
    verify_generator,
)

sp.init_printing()

x, t = sp.symbols("x t")
u = sp.Function("u")(x, t)

# Porous Medium Equation: u_t - (u*u_x)_x = 0
pme_eq = u.diff(t) - u.diff(x) ** 2 - u * u.diff(x, 2)
print("PDE Order:", max_derivative_order(pme_eq, u, (x, t)))
sp.Eq(pme_eq, 0)

## 1. Lie Point Symmetry Generators

We solve for the point symmetries with `infinitesimals`:

In [ ]:
sol = infinitesimals(pme_eq, u, (x, t), ansatz_degree=1)
print(f"Dimension of the symmetry algebra: {sol.dimension}\n")

labels = [
    "X_1 (Space Translation):",
    "X_2 (Time Translation):",
    "X_3 (Kinematic Scaling):",
    "X_4 (Combined x/Field Scaling):",
]

for label, gen in zip(labels, sol.basis):
    is_valid = verify_generator(pme_eq, u, (x, t), gen)
    print(f"{label}")
    print(f"  xi^x = {gen.xi[0]},  xi^t = {gen.xi[1]},  phi^u = {gen.phi[0]}")
    print(f"  Verified invariant: {is_valid}\n")

## 2. Commutator Table

Evaluating the Lie algebra commutator brackets:

In [ ]:
X1, X2, X3, X4 = sol.basis
print("[X_1, X_3] =", lie_bracket(X1, X3, u, (x, t)))
print("[X_2, X_3] =", lie_bracket(X2, X3, u, (x, t)))
print("[X_1, X_4] =", lie_bracket(X1, X4, u, (x, t)))

## 3. The Barenblatt–Pattle Self-Similar Solution

Combining the scaling symmetries as $X = \tfrac{3}{2}X_3 - \tfrac{1}{2}X_4 = x \partial_x + 3t \partial_t - u \partial_u$ yields the similarity variable $\xi = \frac{x}{t^{1/3}}$ and the famous **Barenblatt solution** describing nonlinearly diffusing gas with a sharp advancing front:
$$u(x, t) = \frac{1}{t^{1/3}} \left( C - \frac{x^2}{6 t^{2/3}} \right)_+$$

In [ ]:
C = sp.symbols("C", positive=True)
barenblatt_solution = (1 / t ** sp.Rational(1, 3)) * (
    C - (x**2) / (6 * t ** sp.Rational(2, 3))
)

print("Barenblatt Solution (inside support):")
display(barenblatt_solution)

# Verify that it satisfies the PME identically inside its support
residual = sp.simplify(pme_eq.subs(u, barenblatt_solution).doit())
print("Residual in Porous Medium Equation:", residual)
assert residual == 0
print("Verification: Barenblatt profile solves the Porous Medium Equation!")